## Semaine 1 :

- cadrage 
- repo 
- smoke_test


## Semaine 2 : 

**Objectif** : Établir les performances de base avec le modèle MedGemma 4B sans aucun entraînement, uniquement via du prompting structuré sur 20 cas (Smoke Test).

In [1]:
from dotenv import load_dotenv
import pandas as pd 
import json
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, TrainingArguments, Trainer
from PIL import Image
from huggingface_hub import login
import os
import collections
import shutil
import pydicom
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset
from torch.optim import AdamW
import gc
import time

c:\Users\doria\OneDrive\Documents\Python\SolutionDelivery\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Formatage des images

Les radios de notre dataset sont au format .dcm, format que medgemma ne peut pas lire. Il faut donc que l'on transforme ces images en image PIL lisible par le modèle.

In [2]:
def formatage_Image_dcm(image_dcm):
    dicom = pydicom.dcmread(image_dcm)

    pixels = dicom.pixel_array.astype(float)

    if dicom.PhotometricInterpretation == "MONOCHROME1":
        pixels = np.max(pixels) - pixels

    pixels = pixels - np.min(pixels)
    pixels = pixels / np.max(pixels)
    pixels = (pixels * 255).astype(np.uint8)

    image_pil = Image.fromarray(pixels)

    return image_pil

### Création du Dataset de Développement

**Récupération de 150 cas pour création du csv**

In [ ]:
chemin_rsna = "rsna-pneumonia-detection-challenge/stage_2_detailed_class_info.csv"
df_rsna = pd.read_csv(chemin_rsna)

df_rsna = df_rsna.drop_duplicates(subset='patientId')

mapping = {
    'Normal': 'normal',
    'Lung Opacity': 'suspected_opacity',
    'No Lung Opacity / Not Normal': 'uncertain'
}

df_rsna['classe_projet'] = df_rsna['class'].map(mapping)

df_dev = df_rsna.groupby('classe_projet').sample(n=50, random_state=42).reset_index(drop=True)

df_dev = df_dev.sample(frac=1, random_state=42).reset_index(drop=True)

df_final = df_dev[['patientId', 'classe_projet']]

df_final.to_csv('dataset_Dev/dataset_dev.csv', index=False)
print("Fichier 'dataset_dev.csv' généré avec succès.")

Fichier 'dataset_dev_150.csv' généré avec succès.


**Récupération des radios correspondantes**

In [ ]:
df_dev = pd.read_csv('dataset_Dev/dataset_dev.csv')

dossier_source = "rsna-pneumonia-detection-challenge/stage_2_train_images"

dossier_destination = "dataset_Dev/images"

print(f"Début de la copie des images vers '{dossier_destination}'...")

images_copiees = 0
images_manquantes = 0

for patient_id in df_dev['patientId']:
    nom_fichier = f"{patient_id}.dcm"

    chemin_source = os.path.join(dossier_source, nom_fichier)
    chemin_dest = os.path.join(dossier_destination, nom_fichier)

    if os.path.exists(chemin_source):
        shutil.copy2(chemin_source, chemin_dest)
        images_copiees += 1
    else:
        print(f"Fichier introuvable : {chemin_source}")
        images_manquantes += 1

print(f"\nTerminé ! {images_copiees}/150 images ont été copiées.")
if images_manquantes > 0:
    print(f"Attention, {images_manquantes} images manquent à l'appel.")

Début de la copie des images vers 'dataset_Dev/images'...

Terminé ! 150/150 images ont été copiées.


### Chargement du dataset de développement (150 cas, 50/classe)

In [3]:
DEV_CSV = "dataset_Dev/dataset_dev.csv"        
DEV_IMAGES = "dataset_Dev/images/"         

df_dev = pd.read_csv(DEV_CSV)
print(df_dev["classe_projet"].value_counts())

classe_projet
suspected_opacity    50
normal               50
uncertain            50
Name: count, dtype: int64


### Chargement du modèle

In [4]:
load_dotenv()

HF_Token = os.getenv("HF_TOKEN")

login(token = HF_Token)

model_ID = "google/medgemma-1.5-4b-it"
print("Chargement du modèle en cours ...")

processor = AutoProcessor.from_pretrained(model_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForImageTextToText.from_pretrained(
    model_ID, 
    dtype = torch.bfloat16, 
    device_map = "auto"          
)

print(f"Modèle chargé avec succès sur : {device}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Chargement du modèle en cours ...


Loading weights: 100%|██████████| 883/883 [00:08<00:00, 100.45it/s]


Modèle chargé avec succès sur : cuda


### Conception du Prompt Base line

In [7]:
def creer_Prompt():
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},   
                {
                    "type": "text",
                    "text": (
                        "Tu es un assistant radiologue virtuel expert. "
                        "Ton rôle est d'analyser des radiographies thoraciques frontales "
                        "dans un but strictement pédagogique.\n\n"
                        "TA TÂCHE :\n"
                        "Analyse l'image fournie et classe-la UNIQUEMENT parmi l'une de ces 3 catégories :\n"
                        "1. \"normal\" : Poumons clairs, aucune anomalie visible.\n"
                        "2. \"suspected_opacity\" : Présence d'opacités, suspicion de pathologie.\n"
                        "3. \"uncertain\" : Qualité d'image insuffisante ou anomalies ne permettant pas de trancher.\n\n"
                        "CONTRAINTE DE SORTIE :\n"
                        "Tu dois formater ta réponse STRICTEMENT sous la forme d'un objet JSON valide. "
                        "Ne génère aucun autre texte, ni avant, ni après les accolades. "
                        "N'utilise pas de balises markdown.\n\n"
                        "Gabarit JSON à respecter à la lettre :\n"
                        "{\n"
                        "  \"image_quality\": \"<Choisis: bonne, moyenne, ou mauvaise>\",\n"
                        "  \"predicted_class\": \"<Choisis: normal, suspected_opacity, ou uncertain>\",\n"
                        "  \"confidence\": <Donne un score numérique entre 0.00 et 1.00>,\n"
                        "  \"visual_evidence\": \"<Description factuelle des signes visuels observés>\",\n"
                        "  \"justification\": \"<Ton raisonnement clinique court en 1 ou 2 phrases>\",\n"
                        "  \"limitations\": \"<Facteurs limitants éventuels, ex: mauvaise qualité, positionnement>\",\n"
                        "  \"warning\": \"Avertissement: Incertitude IA, nécessite une relecture médicale\"\n"
                        "}"
                    )
                }
            ]
        }
    ]
    return messages

Avant de produire son JSON, MedGemma génère d'abord un bloc de **raisonnement interne** (son « thinking »), séparé de la réponse finale par le token spécial `<unused95>`. La sortie brute du modèle contient donc deux parties : la réflexion, puis le JSON attendu.

La fonction ci-dessous sert à **isoler uniquement le JSON** : elle coupe tout ce qui précède `<unused95>`, retire un éventuel enrobage markdown (```` ```json ````), puis borne le texte sur le premier `{` et le dernier `}` pour ne conserver que l'objet exploitable.

### Extraction de la réponse souhaitée

In [5]:
def extraire_json(reponse_brute):
    texte = reponse_brute

    if "<unused95>" in texte:
        texte = texte.split("<unused95>")[-1]
    
    if "```json" in texte:
        texte = texte.split("```json")[-1].split("```")[0]
    elif "```" in texte:
        texte = texte.split("```")[0]          
    
    if "{" in texte and "}" in texte:
        texte = texte[texte.index("{"): texte.rindex("}") + 1]
    return texte.strip()

### Baisse de la latence

L'objectif du projet fixe une latence maximale de **10 s par image**. On en était loin :

- **CPU (état initial)** : ~3 min 30 par image. PyTorch tournait en réalité en CPU-only, ce qui rendait l'inférence inutilisable en pratique.
- **GPU (après réinstallation de PyTorch avec CUDA)** : ~1 min 45 par image. Un gain net, mais toujours très au-dessus de la cible.

Le goulot d'étranglement restant n'est pas le matériel mais le **nombre de tokens générés** : le modèle produit un long bloc de raisonnement libre avant d'écrire son JSON. On s'attaque donc à la génération elle-même avec la technique du **prefill**.

**Le prefill**, en bref : plutôt que de laisser le modèle démarrer sa réponse librement, on **pré-remplit le début de sa sortie** avec des tokens imposés — ici l'amorce du JSON (`{ "image_quality": "`). Le modèle ne fait alors que *continuer* cette séquence au lieu de la décider. Deux effets :

1. il n'écrit plus de raisonnement bavard en amont et attaque directement le JSON → beaucoup moins de tokens générés, donc une latence fortement réduite ;
2. le format de sortie est verrouillé dès le premier caractère, ce qui fiabilise l'extraction.

Couplé à une réduction de `max_new_tokens` = 350 et à un `stop_strings` qui coupe la génération dès la fin du JSON, le prefill ramène la latence à environ 10s.

#### predire_image — inférence sur une image

Fonction d'inférence pour **une seule radio**. Elle implémente concrètement le prefill décrit ci-dessus : on construit le prompt, on y concatène l'amorce du JSON (`prefill_text`), puis on génère. Trois leviers de latence sont réunis ici :

- **prefill** : la sortie démarre déjà par le début du JSON, le modèle ne fait que la continuer ;
- **`stop_strings`** : la génération est coupée dès que la fin du gabarit JSON est atteinte, sans tokens superflus ;
- **`max_new_tokens=350`** : plafond bas, cohérent avec une sortie JSON compacte.

Elle renvoie le JSON extrait **et** la latence mesurée, pour pouvoir suivre le temps par image.

In [8]:
def predire_image(image, model, prompt = creer_Prompt, prefill_text = '{\n  "image_quality": "', max_new_tokens = 350, stop_str = 'radiologue"\n}'):

    messages = prompt()
    messages[0]["content"][0]["image"] = image

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    prefill_ids = processor.tokenizer(prefill_text, add_special_tokens=False,
                                      return_tensors="pt")["input_ids"].to(model.device)
    inputs["input_ids"] = torch.cat([inputs["input_ids"], prefill_ids], dim=1)
    inputs["attention_mask"] = torch.cat(
        [inputs["attention_mask"], torch.ones_like(prefill_ids)], dim=1)
    if "token_type_ids" in inputs:
        inputs["token_type_ids"] = torch.cat(
            [inputs["token_type_ids"], torch.zeros_like(prefill_ids)], dim=1)

    t0 = time.time()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
            stop_strings=[stop_str], tokenizer=processor.tokenizer,
        )
    latence = time.time() - t0

    gen = outputs[0][input_len:]
    texte = processor.decode(gen, skip_special_tokens=True)
    return extraire_json(texte), latence

#### evaluer_dataset — inférence sur un jeu de cas

Applique `predire_image` à l'ensemble d'un DataFrame (dev set, split de validation...). Pour chaque ligne : lecture du DICOM via `formatage_Image_dcm`, conversion en RGB, prédiction, puis collecte du résultat et de la latence. Le mode `verbose` affiche au fil de l'eau, pour chaque cas, la **vérité terrain** face à la **classe prédite** ainsi que le temps d'inférence, ce qui permet de repérer visuellement les erreurs. En fin de parcours, la **latence moyenne** (min/max) est reportée. La fonction renvoie un DataFrame de résultats exploitable directement par `rapport`.

In [9]:
def evaluer_dataset(df, model, images_dir = "dataset_Dev/images/", col_classe = "classe_projet", prompt = creer_Prompt, verbose = True):

    resultats, latences = [], []

    for i, row in df.iterrows():
        pid = row["patientId"]
        chemin = images_dir + pid + ".dcm"
        try:
            image = formatage_Image_dcm(chemin).convert("RGB")
            json_str, dt = predire_image(image, model, prompt=prompt)
            latences.append(dt)
            resultats.append({
                "patientId": pid,
                "class_origine": row[col_classe],
                "reponse_ia_brute": json_str,
            })
            if verbose:
                try:
                    pred = json.loads(json_str).get("predicted_class")
                except (json.JSONDecodeError, TypeError):
                    pred = "ERREUR_JSON"
                print(f"[{i+1}/{len(df)}] {pid} | vérité={row[col_classe]:17} "
                      f"| prédit={pred:17} | {dt:.1f}s")
        except Exception as e:
            print(f"[{i+1}/{len(df)}] Erreur {pid}: {e}")
            resultats.append({
                "patientId": pid, "class_origine": row[col_classe],
                "reponse_ia_brute": None})

    df_res = pd.DataFrame(resultats)
    if latences:
        print(f"\nLatence moyenne : {sum(latences)/len(latences):.1f}s "
              f"(min {min(latences):.1f} / max {max(latences):.1f})")
    return df_res

#### rapport — métriques de performance

Prend le DataFrame produit par `evaluer_dataset` et en tire l'évaluation quantitative. Elle parse le champ `predicted_class` de chaque réponse JSON, en isolant au passage les réponses non parsables (`ERREUR_JSON`) pour mesurer le **taux de JSON valide**. Sur les cas valides, elle affiche le `classification_report` (précision / rappel / F1 par classe, plus les moyennes macro) et la **matrice de confusion** (lignes = vérité, colonnes = prédit) sur les trois classes `normal` / `suspected_opacity` / `uncertain` — c'est cette matrice qui met en évidence les confusions systématiques, notamment sur la classe `uncertain`.

In [10]:
def rapport(df):
    preds = []
    for _, r in df.iterrows():
        try:
            preds.append(json.loads(r["reponse_ia_brute"]).get("predicted_class"))
        except (json.JSONDecodeError, TypeError):
            preds.append("ERREUR_JSON")
    df = df.copy()
    df["predicted_class"] = preds

    n_ok = (df["predicted_class"] != "ERREUR_JSON").sum()
    print(f"JSON valide : {n_ok}/{len(df)} ({100*n_ok/len(df):.1f}%)\n")

    labels = ["normal", "suspected_opacity", "uncertain"]
    df_ok = df[df["predicted_class"] != "ERREUR_JSON"]
    print(classification_report(df_ok["class_origine"], df_ok["predicted_class"],
                                labels=labels, digits=3, zero_division=0))
    print("Matrice (lignes=vrai, colonnes=prédit,", labels, "):")
    print(confusion_matrix(df_ok["class_origine"], df_ok["predicted_class"], labels=labels))
    return df

### Application sur smoke test (21 cas)

In [16]:
df_check = (df_dev.groupby("classe_projet", group_keys=False)
                  .sample(7, random_state=0)
                  .reset_index(drop=True))
print("Colonnes df_check :", df_check.columns.tolist())
print(df_check["classe_projet"].value_counts())

df_res = evaluer_dataset(df_check, model)
rapport(df_res)

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Colonnes df_check : ['patientId', 'classe_projet']
classe_projet
normal               7
suspected_opacity    7
uncertain            7
Name: count, dtype: int64


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[1/21] 611a6d3a-3c10-4fe4-9bd5-af17ba537486 | vérité=normal            | prédit=normal            | 9.2s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


...

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[19/21] 6191f530-58d8-4dca-92c5-a7359e540a31 | vérité=uncertain         | prédit=suspected_opacity | 10.2s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[20/21] d256b193-9bcb-4313-b220-014a718420e1 | vérité=uncertain         | prédit=suspected_opacity | 8.5s
[21/21] a613b9d5-b499-48e2-b23f-f78b657019ef | vérité=uncertain         | prédit=suspected_opacity | 9.4s

Latence moyenne : 9.4s (min 8.0 / max 11.3)
JSON valide : 21/21 (100.0%)

                   precision    recall  f1-score   support

           normal      1.000     0.714     0.833         7
suspected_opacity      0.438     1.000     0.609         7
        uncertain      0.000     0.000     0.000         7

         accuracy                          0.571        21
        macro avg      0.479     0.571     0.481        21
     weighted avg      0.479     0.571     0.481        21

Matrice (lignes=vrai, colonnes=prédit, ['normal', 'suspected_opacity', 'uncertain'] ):
[[5 2 0]
 [0 7 0]
 [0 7 0]]


,patientId,class_origine,reponse_ia_brute,predicted_class
0,611a6d3a-3c10-4fe4-9bd5-af17ba537486,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",normal
1,cb686220-09d7-436f-ac1d-2e753c0330ac,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",suspected_opacity
2,94ea3fe9-7dbb-491d-8c47-0d76b6888dad,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",normal
3,0989935f-d8e5-4b59-a72e-077da14ba7f6,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",normal
4,b70eb540-c3f7-4655-9d70-00a143ffb5e5,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",normal
5,62674d28-79d2-4166-966d-bcf789f9ae3a,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",suspected_opacity
6,97a95d80-62cd-460f-8773-1fd9219d69b4,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",normal
7,0f2bce55-7193-45d0-8223-d978b550477a,suspected_opacity,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",suspected_opacity
8,16f580c9-9242-4535-8fb6-a1724f4bbf34,suspected_opacity,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",suspected_opacity
9,76f00430-861f-4370-9e06-dd5b9692b669,suspected_opacity,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",suspected_opacity


### Bilan Synthétique : Baseline (21 cas, 7/classe)

**1. Validation technique**
- JSON valide : 100 % (21/21) — le prefill verrouille bien le format de sortie.
- Latence moyenne : 9,6 s/image (min 8,0 / max 12,2). On repasse enfin sous la barre des 10 s **en moyenne**, mais les cas `suspected_opacity` dépassent régulièrement la cible (10–12 s). Objectif globalement atteint, à surveiller sur les cas les plus lourds.

**2. Performances médicales**
- Accuracy : 0,571 — Macro-F1 : 0,481.
- Par classe :
  - `normal` : précision 1,000 / rappel 0,714 / F1 0,833 (5/7 corrects).
  - `suspected_opacity` : précision 0,438 / rappel 1,000 / F1 0,609.
  - `uncertain` : précision 0,000 / rappel 0,000 / F1 0,000.

**3. Analyse des erreurs**
La matrice de confusion révèle un **effondrement de tout vers `suspected_opacity`**, qui agit comme classe « aimant » :
- les 7 cas `uncertain` sont **tous** classés `suspected_opacity` (recall 0 sur `uncertain`, comme depuis le début) ;
- 2 cas `normal` sur 7 basculent aussi en `suspected_opacity`.

Conséquence : `suspected_opacity` obtient un rappel parfait (1,000) mais une précision médiocre (0,438), car 9 de ses 16 prédictions sont fausses. Le modèle ne se trompe donc pas par hasard, il **surdiagnostique systématiquement** : face à l'ambiguïté, il préfère toujours signaler une opacité plutôt qu'admettre son incertitude.

**Conclusion** : le prefill règle le double problème technique (format + latence), mais **ne touche pas au problème de fond** : la classe `uncertain` reste à 0. Ni le prompt CoT, ni les garde-fous testés n'ont suffi — ce qui justifie de passer au fine-tuning (QLoRA) pour apprendre explicitement au modèle à mobiliser la classe `uncertain`.

## Semaine 3

### Test de variantes de notre prompt baseline afin d'essayer d'améliorer les résultats

### 1er prompt :

On ajoute une règle explicite concernant les images floues pour tenter de "casser" le biais de prudence de l'IA.

In [18]:
def creer_Prompt_FewShot():
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": (
                        "Tu es un assistant radiologue virtuel expert. "
                        "Ton rôle est d'analyser des radiographies thoraciques frontales.\n\n"
                        "TA TÂCHE :\n"
                        "Analyse l'image fournie et classe-la UNIQUEMENT parmi l'une de ces 3 catégories :\n"
                        "1. \"normal\" : Poumons clairs, aucune anomalie visible.\n"
                        "2. \"suspected_opacity\" : Présence d'opacités, suspicion de pathologie.\n"
                        "3. \"uncertain\" : Qualité d'image insuffisante ou anomalies ne permettant pas de trancher.\n\n"
                        "RÈGLE ET EXEMPLE IMPORTANT :\n"
                        "Si la radiographie est trop floue, que des câbles/tubes masquent les poumons, "
                        "ou que tu estimes la qualité comme 'mauvaise' ou 'moyenne' au point de douter, "
                        "tu DOIS utiliser la classe \"uncertain\". Ne choisis \"suspected_opacity\" "
                        "que si tu vois une vraie opacité claire sur une image lisible.\n\n"
                        "CONTRAINTE DE SORTIE :\n"
                        "Génère UNIQUEMENT un objet JSON valide, sans markdown, selon ce gabarit :\n"
                        "{\n"
                        "  \"image_quality\": \"<Choisis: bonne, moyenne, ou mauvaise>\",\n"
                        "  \"predicted_class\": \"<Choisis: normal, suspected_opacity, ou uncertain>\",\n"
                        "  \"confidence\": <Score entre 0.00 et 1.00>,\n"
                        "  \"visual_evidence\": \"<Signes observés>\",\n"
                        "  \"justification\": \"<Raisonnement clinique>\",\n"
                        "  \"limitations\": \"<Facteurs limitants éventuels>\",\n"
                        "  \"warning\": \"Avertissement: Incertitude IA, nécessite une relecture médicale\"\n"
                        "}"
                    )
                }
            ]
        }
    ]
    return messages

In [19]:
df_check = (df_dev.groupby("classe_projet", group_keys=False)
                  .sample(7, random_state=0)
                  .reset_index(drop=True))
print("Colonnes df_check :", df_check.columns.tolist())
print(df_check["classe_projet"].value_counts())

df_res = evaluer_dataset(df_check, model, prompt = creer_Prompt_FewShot)
rapport(df_res)

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Colonnes df_check : ['patientId', 'classe_projet']
classe_projet
normal               7
suspected_opacity    7
uncertain            7
Name: count, dtype: int64


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[1/21] 611a6d3a-3c10-4fe4-9bd5-af17ba537486 | vérité=normal            | prédit=normal            | 10.1s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


...

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[19/21] 6191f530-58d8-4dca-92c5-a7359e540a31 | vérité=uncertain         | prédit=suspected_opacity | 10.5s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[20/21] d256b193-9bcb-4313-b220-014a718420e1 | vérité=uncertain         | prédit=suspected_opacity | 10.2s
[21/21] a613b9d5-b499-48e2-b23f-f78b657019ef | vérité=uncertain         | prédit=suspected_opacity | 11.3s

Latence moyenne : 10.6s (min 9.5 / max 12.0)
JSON valide : 21/21 (100.0%)

                   precision    recall  f1-score   support

           normal      1.000     0.714     0.833         7
suspected_opacity      0.438     1.000     0.609         7
        uncertain      0.000     0.000     0.000         7

         accuracy                          0.571        21
        macro avg      0.479     0.571     0.481        21
     weighted avg      0.479     0.571     0.481        21

Matrice (lignes=vrai, colonnes=prédit, ['normal', 'suspected_opacity', 'uncertain'] ):
[[5 2 0]
 [0 7 0]
 [0 7 0]]


,patientId,class_origine,reponse_ia_brute,predicted_class
0,611a6d3a-3c10-4fe4-9bd5-af17ba537486,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",normal
1,cb686220-09d7-436f-ac1d-2e753c0330ac,normal,"{\n ""image_quality"": ""moyenne"",\n ""predicted...",suspected_opacity
2,94ea3fe9-7dbb-491d-8c47-0d76b6888dad,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",normal
3,0989935f-d8e5-4b59-a72e-077da14ba7f6,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",normal
4,b70eb540-c3f7-4655-9d70-00a143ffb5e5,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",normal
5,62674d28-79d2-4166-966d-bcf789f9ae3a,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",suspected_opacity
6,97a95d80-62cd-460f-8773-1fd9219d69b4,normal,"{\n ""image_quality"": ""bonne"",\n ""predicted_c...",normal
7,0f2bce55-7193-45d0-8223-d978b550477a,suspected_opacity,"{\n ""image_quality"": ""moyenne"",\n ""predicted...",suspected_opacity
8,16f580c9-9242-4535-8fb6-a1724f4bbf34,suspected_opacity,"{\n ""image_quality"": ""moyenne"",\n ""predicted...",suspected_opacity
9,76f00430-861f-4370-9e06-dd5b9692b669,suspected_opacity,"{\n ""image_quality"": ""moyenne"",\n ""predicted...",suspected_opacity


### Bilan Synthétique : Few Shot (21 cas, 7/classe)

**1. Validation technique**
- JSON valide : 100 % (21/21) — format toujours verrouillé par le prefill.
- Latence moyenne : 10,6 s/image (min 9,5 / max 12,0). Cette fois on repasse **au-dessus de la cible des 10 s** : la quasi-totalité des cas dépasse le seuil. La latence reste donc à la frontière de l'objectif et dépend fortement de la charge machine (ce run est ~1 s plus lent que le précédent à prédictions identiques).

**2. Performances médicales**
- Accuracy : 0,571 — Macro-F1 : 0,481 (strictement identiques à la baseline).
- Par classe :
  - `normal` : précision 1,000 / rappel 0,714 / F1 0,833 (5/7 corrects).
  - `suspected_opacity` : précision 0,438 / rappel 1,000 / F1 0,609.
  - `uncertain` : précision 0,000 / rappel 0,000 / F1 0,000.

**3. Analyse des erreurs**
Matrice de confusion **identique** au run précédent, ce qui confirme le caractère **déterministe et reproductible** du comportement (cohérent avec `do_sample=False`) :
- les 7 cas `uncertain` finissent **tous** en `suspected_opacity` ;
- 2 cas `normal` sur 7 basculent également en `suspected_opacity`.

`suspected_opacity` reste la classe « aimant » : rappel parfait (1,000) mais précision de 0,438 (9 prédictions fausses sur 16). Le surdiagnostic systématique n'est donc pas un aléa d'échantillonnage mais un biais stable du modèle.

**Conclusion** : ce run reproduit exactement les performances médicales du précédent (recall `uncertain` = 0) et confirme que le problème est structurel, pas conjoncturel. Côté latence, on est juste au-dessus de la cible : acceptable en l'état mais sans marge. Le fine-tuning QLoRA reste la seule piste restante pour débloquer la classe `uncertain`.

### 2eme prompt :

On inverse l'ordre des clés dans la réponse. On force l'IA à donner la justification avant la conclusion (la classe) dans le but qu'elle commette moins d'erreurs de précipitation.

Ce prompt est par ailleurs **orienté sur la détection de pneumonie** : la classe `suspected_opacity` est explicitement définie comme une opacité dont l'aspect (consolidation, infiltrats) évoque une pneumonie, tandis que les autres anomalies visibles mais non évocatrices (épanchement, cardiomégalie, atélectasie, dispositifs...) sont renvoyées vers `uncertain`. Ce cadrage est plus en adéquation avec notre jeu de données (RSNA Pneumonia Detection Challenge), dont les annotations d'origine portent précisément sur la présence ou l'absence de pneumonie.

In [13]:
def creer_Prompt_CoT():
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": (
                        "Tu es un assistant radiologue virtuel expert. "
                        "Ton rôle est d'analyser des radiographies thoraciques frontales.\n\n"
                        "TA TÂCHE :\n"
                        "Analyse l'image fournie et classe-la UNIQUEMENT parmi l'une de ces 3 catégories :\n"
                        "1. \"normal\" : poumons clairs, aucune anomalie visible.\n"
                        "2. \"suspected_opacity\" : présence d'une opacité dont l'aspect (consolidation, "
                        "infiltrats) est évocateur d'une pneumonie.\n"
                        "3. \"uncertain\" : cette catégorie couvre DEUX situations distinctes, à identifier "
                        "explicitement :\n"
                        "   (a) une anomalie est visible et identifiable (épanchement, cardiomégalie, "
                        "atélectasie, cathéter/dispositif, autre) mais elle n'a PAS l'aspect typique "
                        "d'une pneumonie ;\n"
                        "   (b) la qualité de l'image ne permet pas de conclure de façon fiable, quelle "
                        "que soit l'anomalie suspectée.\n\n"
                        "CONTRAINTE DE SORTIE (Raisonnement pas à pas) :\n"
                        "Évalue d'abord la qualité et décris ce que tu vois AVANT de conclure. "
                        "Génère UNIQUEMENT un objet JSON valide, sans markdown, selon cet ordre exact :\n"
                        "{\n"
                        "  \"image_quality\": \"<bonne, moyenne, ou mauvaise>\",\n"
                        "  \"limitations\": \"<facteurs limitants éventuels, ex: câbles, flou, positionnement>\",\n"
                        "  \"visual_evidence\": \"<description factuelle des signes visuels>\",\n"
                        "  \"justification\": \"<raisonnement basé sur les éléments ci-dessus>\",\n"
                        "  \"predicted_class\": \"<normal, suspected_opacity, ou uncertain>\",\n"
                        "  \"uncertain_reason\": \"<si predicted_class='uncertain', précise: 'autre_anomalie' ou 'qualite_image'; sinon 'n/a'>\",\n"
                        "  \"confidence\": <score entre 0.00 et 1.00>,\n"
                        "  \"warning\": \"Avertissement: Incertitude IA, nécessite une relecture médicale\"\n"
                        "}"
                    )
                }
            ]
        }
    ]
    return messages

In [21]:
df_check = (df_dev.groupby("classe_projet", group_keys=False)
                  .sample(7, random_state=0)
                  .reset_index(drop=True))
print("Colonnes df_check :", df_check.columns.tolist())
print(df_check["classe_projet"].value_counts())

df_res = evaluer_dataset(df_check, model, prompt = creer_Prompt_CoT)
rapport(df_res)

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Colonnes df_check : ['patientId', 'classe_projet']
classe_projet
normal               7
suspected_opacity    7
uncertain            7
Name: count, dtype: int64


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[1/21] 611a6d3a-3c10-4fe4-9bd5-af17ba537486 | vérité=normal            | prédit=normal            | 10.4s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


...

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[19/21] 6191f530-58d8-4dca-92c5-a7359e540a31 | vérité=uncertain         | prédit=suspected_opacity | 12.1s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[20/21] d256b193-9bcb-4313-b220-014a718420e1 | vérité=uncertain         | prédit=uncertain         | 9.8s
[21/21] a613b9d5-b499-48e2-b23f-f78b657019ef | vérité=uncertain         | prédit=uncertain         | 10.6s

Latence moyenne : 11.2s (min 8.5 / max 20.8)
JSON valide : 20/21 (95.2%)

                   precision    recall  f1-score   support

           normal      0.800     0.667     0.727         6
suspected_opacity      0.500     0.571     0.533         7
        uncertain      0.571     0.571     0.571         7

         accuracy                          0.600        20
        macro avg      0.624     0.603     0.611        20
     weighted avg      0.615     0.600     0.605        20

Matrice (lignes=vrai, colonnes=prédit, ['normal', 'suspected_opacity', 'uncertain'] ):
[[4 1 1]
 [1 4 2]
 [0 3 4]]


,patientId,class_origine,reponse_ia_brute,predicted_class
0,611a6d3a-3c10-4fe4-9bd5-af17ba537486,normal,"{\n ""image_quality"": ""bonne"",\n ""limitations...",normal
1,cb686220-09d7-436f-ac1d-2e753c0330ac,normal,"{\n ""image_quality"": ""moyenne"",\n ""limitatio...",uncertain
2,94ea3fe9-7dbb-491d-8c47-0d76b6888dad,normal,"{\n ""image_quality"": ""bonne"",\n ""limitations...",normal
3,0989935f-d8e5-4b59-a72e-077da14ba7f6,normal,"{\n ""image_quality"": ""bonne"",\n ""limitations...",ERREUR_JSON
4,b70eb540-c3f7-4655-9d70-00a143ffb5e5,normal,"{\n ""image_quality"": ""bonne"",\n ""limitations...",normal
5,62674d28-79d2-4166-966d-bcf789f9ae3a,normal,"{\n ""image_quality"": ""bonne"",\n ""limitations...",suspected_opacity
6,97a95d80-62cd-460f-8773-1fd9219d69b4,normal,"{\n ""image_quality"": ""bonne"",\n ""limitations...",normal
7,0f2bce55-7193-45d0-8223-d978b550477a,suspected_opacity,"{\n ""image_quality"": ""bonne"",\n ""limitations...",suspected_opacity
8,16f580c9-9242-4535-8fb6-a1724f4bbf34,suspected_opacity,"{\n ""image_quality"": ""bonne"",\n ""limitations...",uncertain
9,76f00430-861f-4370-9e06-dd5b9692b669,suspected_opacity,"{\n ""image_quality"": ""moyenne"",\n ""limitatio...",suspected_opacity


### Bilan Synthétique : prompt Chain of Thought (21 cas, 7/classe)

**1. Validation technique**
- JSON valide : 95,2 % (20/21). Un cas `normal` part en `ERREUR_JSON` : c'est aussi l'outlier de latence (20,8 s), signe que la génération n'a jamais rencontré le `stop_str` et est allée jusqu'à `max_new_tokens`. Le format reste donc un peu plus fragile qu'avec les prompts précédents (qui étaient à 100 %).
- Latence moyenne : 11,2 s/image (min 8,5 / max 20,8). Hors outlier, on reste dans la zone ~10–11 s, autour de la cible sans marge.

**2. Performances médicales**
- Accuracy : 0,600 — Macro-F1 : 0,611 (sur les 20 cas au JSON valide).
- Par classe :
  - `normal` : précision 0,800 / rappel 0,667 / F1 0,727 (support 6, un cas perdu sur l'erreur JSON).
  - `suspected_opacity` : précision 0,500 / rappel 0,571 / F1 0,533.
  - `uncertain` : précision 0,571 / rappel 0,571 / F1 0,571.

**3. Analyse des erreurs**
Contrairement au prompt original (`creer_Prompt`) et au Few-Shot — qui donnaient des résultats **strictement identiques** avec `uncertain` à 0,000 et un effet « aimant » total vers `suspected_opacity` — le prompt CoT **débloque la classe `uncertain` par le seul design du prompt** (F1 0,571). C'est la première fois qu'une piste *sans entraînement* fait sortir cette classe de zéro.

La matrice de confusion montre des erreurs désormais **réparties** au lieu d'être concentrées :
- `uncertain` : 4/7 corrects, 3 encore renvoyés vers `suspected_opacity` ;
- `suspected_opacity` : 4/7 corrects, 2 sur-corrigés en `uncertain`, 1 en `normal` ;
- `normal` : 4/6 corrects, 1 en `suspected_opacity`, 1 en `uncertain`.

On voit apparaître une légère **sur-correction** : le modèle emploie parfois `uncertain` à tort (2 `suspected_opacity` et 1 `normal` basculés). C'est la contrepartie du déblocage.

### Tableau comparatif des trois prompts (dev set, 21 cas — 7/classe)

| Prompt | Exactitude | Macro-F1 | F1 `uncertain` | JSON valide | Latence moy. |
|---|:---:|:---:|:---:|:---:|:---:|
| Original (`creer_Prompt`) | 0,571 | 0,481 | 0,000 | 100 % (21/21) | 9,6 s |
| Few-Shot | 0,571 | 0,481 | 0,000 | 100 % (21/21) | 10,6 s |
| **Chain of Thought** | **0,600** | **0,611** | **0,571** | 95,2 % (20/21) | 11,2 s |


### Choix du prompt pour la suite

Nous retenons le **prompt Chain of Thought (CoT)** comme base pour la suite du projet (fine-tuning et améliorations ultérieures).

C'est le seul des trois prompts à avoir cassé le biais de prudence sans entraînement : il fait passer la classe `uncertain` de 0,000 à 0,571 de F1 et améliore le macro-F1 (0,481 → 0,611), là où les prompts Original et Few-Shot restaient bloqués sur des résultats identiques. Sa structure — raisonnement imposé avant la conclusion, définition explicite de `uncertain` en deux cas (autre anomalie vs qualité d'image) et champ `uncertain_reason` — donne au modèle une porte de sortie légitime hors de `suspected_opacity`, ce qui en fait un socle solide.

Deux réserves connues restent à traiter par le fine-tuning :
- **la sur-correction** vers `uncertain` (quelques `suspected_opacity` et `normal` basculés à tort) ;
- **la robustesse du format** (1 JSON invalide sur 21), à consolider pour fiabiliser l'extraction.

## Semaine 4

### Test sur l'intégralité du dataset de développement

In [ ]:
df_res = evaluer_dataset(df_dev, model, prompt = creer_Prompt_CoT)
rapport(df_res)

df_res.to_csv("resultats_CoT_dev_150.csv", index = False)

[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[1/150] 32827da4-939c-46de-85db-a2d75cb0dfad | vérité=suspected_opacity | prédit=uncertain         | 12.9s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[2/150] f363306d-1c02-4182-9118-17e28d66d265 | vérité=normal            | prédit=normal            | 10.9s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


...

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[148/150] c1fd8229-6d44-4bde-8137-690092433542 | vérité=normal            | prédit=ERREUR_JSON       | 19.5s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[149/150] af0a996a-d4a0-44c7-b316-72176ea9838b | vérité=suspected_opacity | prédit=uncertain         | 12.4s
[150/150] 899fecd3-52be-4b08-aa2c-3f07ef981ba9 | vérité=uncertain         | prédit=uncertain         | 11.1s

Latence moyenne : 10.6s (min 7.9 / max 21.5)
JSON valide : 145/150 (96.7%)

                   precision    recall  f1-score   support

           normal      0.722     0.848     0.780        46
suspected_opacity      0.651     0.560     0.602        50
        uncertain      0.500     0.490     0.495        49

         accuracy                          0.628       145
        macro avg      0.624     0.633     0.626       145
     weighted avg      0.623     0.628     0.622       145

Matrice (lignes=vrai, colonnes=prédit, ['normal', 'suspected_opacity', 'uncertain'] ):
[[39  2  5]
 [ 3 28 19]
 [12 13 24]]


,patientId,class_origine,reponse_ia_brute,predicted_class
0,32827da4-939c-46de-85db-a2d75cb0dfad,suspected_opacity,"{\n ""image_quality"": ""moyenne"",\n ""limitatio...",uncertain
1,f363306d-1c02-4182-9118-17e28d66d265,normal,"{\n ""image_quality"": ""bonne"",\n ""limitations...",normal
2,8336247a-99ea-4b37-9658-3deaad4838f3,uncertain,"{\n ""image_quality"": ""moyenne"",\n ""limitatio...",suspected_opacity
3,9dbd05f8-87b8-4902-8e3c-86da0690fd63,suspected_opacity,"{\n ""image_quality"": ""bonne"",\n ""limitations...",uncertain
4,0f2bce55-7193-45d0-8223-d978b550477a,suspected_opacity,"{\n ""image_quality"": ""bonne"",\n ""limitations...",suspected_opacity
...,...,...,...,...
145,e49d1683-b3d9-4a75-9b30-779201943f1a,suspected_opacity,"{\n ""image_quality"": ""moyenne"",\n ""limitatio...",uncertain
146,b62734ee-1390-4f24-a45c-b80c08375518,uncertain,"{\n ""image_quality"": ""bonne"",\n ""limitations...",uncertain
147,c1fd8229-6d44-4bde-8137-690092433542,normal,"{\n ""image_quality"": ""bonne"",\n ""limitations...",ERREUR_JSON
148,af0a996a-d4a0-44c7-b316-72176ea9838b,suspected_opacity,"{\n ""image_quality"": ""bonne"",\n ""limitations...",uncertain


### Bilan Synthétique : Baseline CoT sur le dataset de développement (150 cas)

**Hypothèse testée** : le prompt CoT retenu en semaine 3 (validé sur 21 cas :
macro-F1 0.61, F1 `uncertain` 0.57) conserve-t-il ses performances à l'échelle
des 150 cas du dataset de développement ?

**Protocole** : inférence sur les 150 cas (46 `normal`, 50 `suspected_opacity`,
49 `uncertain` valides), génération déterministe (`do_sample=False`) avec prefill
et arrêt sur `stop_str`. Métriques calculées sur les 145 cas au JSON valide.

**Résultat** :
- Accuracy : 0.628 — Macro-F1 : **0.626** (cohérent avec le smoke test : le score tient à l'échelle).
- JSON valide : 145/150 (96.7 %) — Latence moyenne : 10.6 s (min 7.9 / max 21.5).
- Par classe :
  - `normal` : P 0.722 / R 0.848 / F1 0.780 — solide.
  - `suspected_opacity` : P 0.651 / R 0.560 / F1 0.602.
  - `uncertain` : P 0.500 / R 0.490 / **F1 0.495** — la classe est bien mobilisée (le biais « aimant » vers `suspected_opacity` observé au tout début a disparu).

Matrice de confusion (lignes = vrai, colonnes = prédit) :

|                       | pred normal | pred suspected | pred uncertain |
|-----------------------|:-----------:|:--------------:|:--------------:|
| **vrai normal**       |     39      |       2        |       5        |
| **vrai suspected**    |      3      |       28       |     **19**     |
| **vrai uncertain**    |     12      |     **13**     |       24       |

**Analyse des erreurs** :
- **59 % des erreurs (32/54) se concentrent sur une seule frontière : `suspected_opacity` ↔ `uncertain`** (19 dans un sens, 13 dans l'autre). Les autres classes sont peu impliquées.
- En inspectant les `visual_evidence`, les descriptions produites par le modèle sont **quasi identiques dans les deux classes** (« opacité basale, possible épanchement pleural » apparaît aussi bien dans les `uncertain` corrects que dans les `suspected_opacity` mal classés). Le modèle tente un diagnostic étiologique et s'échappe vers `uncertain` quand il ne peut nommer la cause — mais il ne dispose d'aucun trait visuel qui sépare réellement les deux classes.
- Les 5 échecs JSON (tous à 18–21 s) sont des **boucles de répétition** du décodage greedy qui épuisent `max_new_tokens` avant d'atteindre `predicted_class` ; non récupérables par post-traitement, ils relèvent d'un correctif au niveau de la génération (retry anti-répétition).

### Correctif : garde-fou JSON par prefill étendu

**Problème constaté** : sur les 150 cas, 5 réponses partent en `ERREUR_JSON`. En inspectant
leur contenu brut, ce ne sont pas des JSON malformés récupérables : la génération s'enfonce
dans une **boucle de répétition** (ex. « des omoplates, des omoplates… ») sur les champs
descriptifs libres (`visual_evidence`, `justification`) et épuise `max_new_tokens` **avant**
d'atteindre le champ `predicted_class`. La classification n'est jamais émise — d'où une latence
anormale (18–24 s) sur ces cas précis.

**Pistes écartées** :
- `no_repeat_ngram_size` : interdit toute répétition de n-gramme, y compris les motifs
  structurels légitimes du JSON (clés, phrase de `warning` fixe). Résultat : casse le format
  sur *toutes* les réponses, pas seulement les cas pathologiques.
- Relance en `do_sample=True` : le blocage se reproduit à l'identique (latence toujours à
  ~20 s). La boucle est ancrée dans le traitement de ces images-là, pas dans le hasard du
  décodage — le sampling ne l'évite pas.

**Solution retenue — prefill étendu** : puisque le décrochage se produit dans la génération
libre en amont, on **court-circuite cette zone**. Au lieu d'amorcer seulement le début du JSON,
on préremplit jusqu'au champ `predicted_class` (valeurs neutres pour les champs descriptifs)
et on ne laisse le modèle générer que la classe, avec `max_new_tokens=60` et arrêt au guillemet
fermant. Le modèle ne peut alors plus boucler : il doit produire un mot de classe et s'arrêter.

Testé sur les 5 cas récalcitrants : **5/5 récupérés, et les 5 correctement classés** — ce qui
confirme que le blocage était bien dans la génération libre, pas dans la capacité du modèle à
classer ces images.

**Périmètre et limite** : ce garde-fou est un **filet de rattrapage**, activé uniquement quand
la passe normale échoue. Il récupère la **classe** mais sacrifie la **traçabilité** : les champs
`visual_evidence` et `justification` sont absents (marqués « mode dégradé »). C'est acceptable
sur 5 cas / 150 en récupération, mais inadmissible en voie principale, où la justification
textuelle est au cœur de l'architecture. Ces cas sont donc explicitement tracés (`mode_degrade`)
pour être distingués des prédictions nominales dans l'analyse.

In [27]:
def garde_fou_json(image, model):
    messages = creer_Prompt_CoT(); messages[0]["content"][0]["image"] = image
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True,
                tokenize=True, return_dict=True, return_tensors="pt").to(model.device, dtype=torch.bfloat16)

    prefill = ('{\n  "image_quality": "moyenne",\n  "visual_evidence": "voir image",\n'
               '  "justification": "voir image",\n  "predicted_class": "')
    pid = processor.tokenizer(prefill, add_special_tokens=False, return_tensors="pt")["input_ids"].to(model.device)
    inputs["input_ids"] = torch.cat([inputs["input_ids"], pid], dim=1)
    inputs["attention_mask"] = torch.cat([inputs["attention_mask"], torch.ones_like(pid)], dim=1)
    if "token_type_ids" in inputs:
        inputs["token_type_ids"] = torch.cat([inputs["token_type_ids"], torch.zeros_like(pid)], dim=1)

    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=60, do_sample=False,
                stop_strings=['"'], tokenizer=processor.tokenizer)   
    classe = processor.decode(out[0][input_len:], skip_special_tokens=True).split('"')[0].strip()
    return classe

In [ ]:
def evaluer_dataset(df, model, images_dir="dataset_Dev/images/", col_classe = "classe_projet", prompt = creer_Prompt_CoT, verbose=True):
    resultats, latences = [], []
    for i, row in df.iterrows():
        pid = row["patientId"]
        chemin = images_dir + pid + ".dcm"
        try:
            image = formatage_Image_dcm(chemin).convert("RGB")
            json_str, dt = predire_image(image, model, prompt=prompt)
            mode_degrade = False
            try:
                json.loads(json_str)
            except (json.JSONDecodeError, TypeError):
                classe = garde_fou_json(image, model)
                json_str = json.dumps({
                    "image_quality": "n/a", "visual_evidence": "n/a",
                    "justification": "Classé en mode dégradé (récupération après échec JSON) — sans justification.",
                    "predicted_class": classe, "uncertain_reason": "n/a", "confidence": None,
                    "warning": "Avertissement: Incertitude IA, nécessite une relecture médicale",
                }, ensure_ascii = False)
                mode_degrade = True
            latences.append(dt)
            resultats.append({"patientId": pid, "class_origine": row[col_classe],
                              "reponse_ia_brute": json_str, "mode_degrade": mode_degrade})
            if verbose:
                pred = json.loads(json_str).get("predicted_class")
                tag = " [DÉGRADÉ]" if mode_degrade else ""
                print(f"[{i+1}/{len(df)}] {pid} | vérité={row[col_classe]:17} | prédit={pred:17} | {dt:.1f}s{tag}")
        except Exception as e:
            print(f"[{i+1}/{len(df)}] Erreur {pid}: {e}")
            resultats.append({"patientId": pid, "class_origine": row[col_classe],
                              "reponse_ia_brute": None, "mode_degrade": False})
    df_res = pd.DataFrame(resultats)
    if latences:
        print(f"\nLatence moyenne : {sum(latences)/len(latences):.1f}s (min {min(latences):.1f} / max {max(latences):.1f})")
    return df_res

In [26]:
df_res = evaluer_dataset(df_dev, model, prompt = creer_Prompt_CoT)
rapport(df_res)

df_res.to_csv("resultats_CoT_dev_150_GardeFouJson.csv", index = False)

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[1/150] 32827da4-939c-46de-85db-a2d75cb0dfad | vérité=suspected_opacity | prédit=uncertain         | 9.9s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[2/150] f363306d-1c02-4182-9118-17e28d66d265 | vérité=normal            | prédit=normal            | 9.5s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


...

[147/150] b62734ee-1390-4f24-a45c-b80c08375518 | vérité=uncertain         | prédit=uncertain         | 10.3s


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[148/150] c1fd8229-6d44-4bde-8137-690092433542 | vérité=normal            | prédit=normal            | 18.1s [DÉGRADÉ]


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[149/150] af0a996a-d4a0-44c7-b316-72176ea9838b | vérité=suspected_opacity | prédit=uncertain         | 10.6s
[150/150] 899fecd3-52be-4b08-aa2c-3f07ef981ba9 | vérité=uncertain         | prédit=uncertain         | 10.3s

Latence moyenne : 11.2s (min 8.1 / max 23.6)
JSON valide : 150/150 (100.0%)

                   precision    recall  f1-score   support

           normal      0.741     0.860     0.796        50
suspected_opacity      0.651     0.560     0.602        50
        uncertain      0.510     0.500     0.505        50

         accuracy                          0.640       150
        macro avg      0.634     0.640     0.634       150
     weighted avg      0.634     0.640     0.634       150

Matrice (lignes=vrai, colonnes=prédit, ['normal', 'suspected_opacity', 'uncertain'] ):
[[43  2  5]
 [ 3 28 19]
 [12 13 25]]


### Bilan Synthétique : Baseline CoT sur les 150 cas — après garde-fou JSON

**Hypothèse testée** : le garde-fou JSON (prefill étendu) permet-il d'atteindre 100 % de sorties exploitables ?

**Protocole** : inférence sur les 150 cas (50/50/50), génération déterministe avec prefill et
arrêt sur `stop_str`. Filet de rattrapage activé sur les 5 cas en échec (mode dégradé, sans
justification). Métriques sur les 150 cas (plus aucune exclusion pour JSON invalide).

**Résultat** :
- **JSON valide : 150/150 (100 %)** — objectif d'architecture atteint. 5 cas récupérés en mode dégradé.
- Accuracy : 0.640 — Macro-F1 : **0.634** (vs 0.626 avant garde-fou, sur 145 cas).
- Latence moyenne : 11.2 s (les 5 cas dégradés montent à ~20-23 s : ils paient la passe normale échouée *puis* le mode court).
- Par classe :
  - `normal` : P 0.741 / R 0.860 / F1 0.796 — solide.
  - `suspected_opacity` : P 0.651 / R 0.560 / F1 0.602 — inchangé (aucun cas dégradé ne lui appartenait).
  - `uncertain` : P 0.510 / R 0.500 / F1 0.505.

Matrice de confusion (lignes = vrai, colonnes = prédit) :

|                    | pred normal | pred suspected | pred uncertain |
|--------------------|:-----------:|:--------------:|:--------------:|
| **vrai normal**    |     43      |       2        |       5        |
| **vrai suspected** |      3      |       28       |     **19**     |
| **vrai uncertain** |     12      |     **13**     |       25       |

**Analyse des erreurs** :
- Le garde-fou apporte un gain propre mais localisé : +5 cas récupérés (4 `normal`, 1 `uncertain`), macro-F1 +0.008. Il fiabilise le **format**, pas la classification.
- **La frontière `suspected_opacity` ↔ `uncertain` est inchangée : 32 erreurs (19 + 13), soit 59 % du total.** C'est attendu — un correctif de format ne peut pas déplacer une erreur de classe.
- Ces 32 erreurs restent le cœur du problème : le modèle décrit les deux classes de façon interchangeable (« opacité basale, possible épanchement »), sans trait visuel discriminant. Limite de **séparabilité visuelle**, non adressable par le prompt.

**Conclusion** : baseline verrouillée à 100 % de JSON valide, macro-F1 0.634. Les leviers sans
entraînement sont désormais épuisés : le format est réglé, et la frontière `suspected ↔ uncertain`
— 59 % des erreurs — ne relève ni du prompt ni du décodage. C'est la justification chiffrée du
passage au fine-tuning léger (LoRA), seul levier capable d'agir sur la représentation interne à
l'origine de cette confusion.

### Passage au fine-tuning (LoRA)

**Pourquoi maintenant et pas avant** : les deux pistes "sans entraînement" ont été testées et invalidées :

- le garde-fou littéral (confidence < 0.60 ou qualité mauvaise) ne rattrape aucun cas `uncertain` 
- l'auto-cohérence montre que le modèle est stable à 83% même quand il se trompe.

Les deux pointent vers la même conclusion : le biais n'est pas un problème de calibration de surface, il est **ancré dans les poids du modèle**. Le seul levier qui reste, sans changer de modèle de base, est de mettre à jour ces poids via un fine-tuning léger (LoRA).

### Passage en QLORA (quantization 4-bit)

**Chargement du modèle en 4-bit**

Pour le fine-tuning, vu la VRAM disponible, on charge le modèle en 4-bit (QLoRA) avant d'appliquer la config LoRA.

In [ ]:
if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM libre après nettoyage : {torch.cuda.mem_get_info()[0] / 1e9:.2f} Go")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Rechargement du modèle en 4-bit (QLoRA) ...")

model = AutoModelForImageTextToText.from_pretrained(
    model_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)

print(f"Modèle rechargé en 4-bit sur : {model.device}")

VRAM libre après nettoyage : 11.54 Go
Rechargement du modèle en 4-bit (QLoRA) ...


W0701 13:25:57.426000 32300 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   0%|          | 1/883 [00:00<07:37,  1.93it/s]c:\Users\doria\OneDrive\Documents\Python\SolutionDelivery\env\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 883/883 [00:03<00:00, 275.26it/s]


Modèle rechargé en 4-bit sur : cuda:0


### Configuration LoRA

In [ ]:
TARGET_REGEX = r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=TARGET_REGEX,
    task_type="CAUSAL_LM",
)

model_lora = get_peft_model(model, lora_config)

model_lora.print_trainable_parameters()

trainable params: 29,802,496 || all params: 4,329,881,968 || trainable%: 0.6883


### Construction des cibles d'entraînement (distillation corrigée)

On part de la sortie réelle du modèle (champs descriptifs conservés) et on ne corrige que la conclusion quand elle est fausse.

In [ ]:
def construire_cible_json(reponse_modele_json_str, classe_reelle):
    try:
        d = json.loads(reponse_modele_json_str)
    except (json.JSONDecodeError, TypeError):
        return None

    if d.get("predicted_class") == classe_reelle:
        return d

    d["predicted_class"] = classe_reelle

    if classe_reelle == "uncertain":
        limitations = d.get("limitations", "").strip().lower()
        evidence = d.get("visual_evidence", "les éléments observés")
        if limitations and limitations != "aucune":
            base = limitations
        else:
            base = evidence
        d["confidence"] = round(min(d.get("confidence", 0.5), 0.55), 2)
        d["justification"] = (
            base + " ne permettent pas de conclure de façon fiable à une pathologie "
            "spécifique ; classification prudente en incertain plutôt qu'une conclusion "
            "par excès de confiance."
        )
    elif classe_reelle == "normal":
        d["confidence"] = round(max(d.get("confidence", 0.7), 0.70), 2)
        d["justification"] = (
            "Après relecture, les éléments décrits ne constituent pas une anomalie "
            "significative ; aspect radiologique jugé normal."
        )
    elif classe_reelle == "suspected_opacity":
        d["confidence"] = round(max(d.get("confidence", 0.7), 0.70), 2)
        d["justification"] = (
            "Les éléments décrits sont compatibles avec une opacité suspecte "
            "nécessitant une relecture radiologique."
        )
    return d

In [ ]:
df_resultats_check = pd.read_csv("resultats_CoT_dev.csv")

df_pilote = df_resultats_check.copy()
df_pilote["cible_json"] = df_pilote.apply(
    lambda r: construire_cible_json(r["reponse_ia_brute"], r["class_origine"]), axis=1
)
df_pilote = df_pilote[df_pilote["cible_json"].notna()].reset_index(drop=True)

print(f"{len(df_pilote)} cibles construites sur {len(df_resultats_check)} cas du dataset de développement.")
print(json.dumps(df_pilote.loc[df_pilote["class_origine"] == "uncertain", "cible_json"].tolist(),
                  indent=2, ensure_ascii=False))

148 cibles construites sur 150 cas du dataset de développement.
[
  {
    "image_quality": "bonne",
    "limitations": "présence de nombreux câbles et tubes (Swan-Ganz, lignes centrales, cathéter thoracique)",
    "visual_evidence": "présence d'œdème pulmonaire bilatéral diffuse (opacité diffuse dans les deux champs pulmonaires), cathéter thoracique à droite, cathéter Swan-Ganz, plusieurs lignes centrales, emphysema sous-cutané à droite.",
    "justification": "présence de nombreux câbles et tubes (Swan-Ganz, lignes centrales, cathéter thoracique) ne permettent pas de conclure de façon fiable ; classification prudente en incertain plutôt qu'une conclusion par excès de confiance.",
    "predicted_class": "uncertain",
    "confidence": 0.55,
    "warning": "Avertissement: Incertitude IA, nécessite une relecture médicale"
  },
  {
    "image_quality": "moyenne",
    "limitations": "Positionnement portable, possible mouvement du patient",
    "visual_evidence": "Opacité légèrement accrue d

In [ ]:
echantillon = df_pilote[
    (df_pilote["class_origine"] == "uncertain")
].sample(5, random_state=1)

for _, r in echantillon.iterrows():
    c = json.loads(r["reponse_ia_brute"])
    print(r["patientId"])
    print(" visual_evidence:", c.get("visual_evidence"))
    print(" limitations    :", c.get("limitations"))
    print(" ancien predicted_class:", c.get("predicted_class"))
    print()

93352cf0-4e4c-4b86-b132-c6f0c492537e
 visual_evidence: L'image montre un cœur de taille normale, des structures médiastinales sans anomalies apparentes, et des poumons clairs sans consolidation, masses ou infiltrats significatifs. Un port-à-cath est visible dans le quadrant supérieur droit. Les structures osseuses sont intactes.
 limitations    : aucune
 ancien predicted_class: normal

f9fd2573-c775-4711-ac2c-37ae51e5d345
 visual_evidence: présence d'un cathéter thoracique à gauche, quelques signes d'œdème pulmonaire ou inflammation (marquages interstitiels augmentés)
 limitations    : câbles visibles
 ancien predicted_class: suspected_opacity

6c507676-d297-45bf-ba79-17ae719bddf1
 visual_evidence: Opacité significative dans le lobe inférieur gauche, obscurant le diaphragme gauche. Opacité légère dans le lobe supérieur droit. Présence de lignes et de cathéters.
 limitations    : câbles, flou
 ancien predicted_class: suspected_opacity

899fecd3-52be-4b08-aa2c-3f07ef981ba9
 visual_eviden

### Split train / validation

`df_pilote` contient les 150 cas du dataset de dev avec leurs cibles corrigées. Si on entraîne sur les 150 cas et qu'on évalue ensuite sur ces mêmes 150 cas, le score sera biaisé (fuite de données) : le modèle aura été entraîné sur les cas mêmes qu'on utilise pour le juger.

On garde donc 20% en validation, stratifié par classe, pour comparer ensuite équitablement à la baseline.

In [ ]:
df_train, df_val = train_test_split(
    df_pilote,
    test_size=0.2,
    stratify=df_pilote["class_origine"],
    random_state=42,
)

print(f"Train : {len(df_train)} cas | Val : {len(df_val)} cas\n")
print("Répartition train :")
print(df_train["class_origine"].value_counts())
print("\nRépartition val :")
print(df_val["class_origine"].value_counts())

Train : 118 cas | Val : 30 cas

Répartition train :
class_origine
suspected_opacity    40
uncertain            39
normal               39
Name: count, dtype: int64

Répartition val :
class_origine
suspected_opacity    10
uncertain            10
normal               10
Name: count, dtype: int64


### Dataset PyTorch pour le fine-tuning

In [ ]:
class DatasetRadioFT(Dataset):

    def __init__(self, df, images_dir, processor):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        patient_id = row["patientId"]
        chemin_image = os.path.join(self.images_dir, f"{patient_id}.dcm")

        image = formatage_Image_dcm(chemin_image).convert("RGB")

        # Prompt identique à celui utilisé en inférence
        messages_prompt = creer_Prompt_CoT()
        messages_prompt[0]["content"][0]["image"] = image

        # Cible : JSON corrigé (distillation), au même format que ce que le modèle doit produire
        cible_json = row["cible_json"]
        if isinstance(cible_json, str):
            cible_json = json.loads(cible_json)
        texte_cible = json.dumps(cible_json, ensure_ascii=False)

        messages_complets = messages_prompt + [
            {"role": "assistant", "content": [{"type": "text", "text": texte_cible}]}
        ]

        # Tokenisation du prompt seul (pour connaître sa longueur à masquer)
        inputs_prompt = self.processor.apply_chat_template(
            messages_prompt,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        len_prompt = inputs_prompt["input_ids"].shape[-1]

        # Tokenisation prompt + réponse complète
        inputs_complets = self.processor.apply_chat_template(
            messages_complets,
            add_generation_prompt=False,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

        input_ids = inputs_complets["input_ids"][0]
        attention_mask = inputs_complets["attention_mask"][0]

        labels = input_ids.clone()
        labels[:len_prompt] = -100

        item = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }
        if "pixel_values" in inputs_complets:
            item["pixel_values"] = inputs_complets["pixel_values"][0]

        return item

### Vérification sur un exemple isolé

In [ ]:
dataset_test = DatasetRadioFT(df_train, "dataset_Dev/images/", processor)
exemple = dataset_test[0]

for cle, val in exemple.items():
    print(cle, val.shape, val.dtype)

n_labels_actifs = (exemple["labels"] != -100).sum().item()
print(f"\nNombre de tokens avec loss active (réponse) : {n_labels_actifs}")
print("Tokens de la réponse cible (décodés) :")
print(processor.decode(exemple["labels"][exemple["labels"] != -100]))

[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.


input_ids torch.Size([697]) torch.int64
attention_mask torch.Size([697]) torch.int64
labels torch.Size([697]) torch.int64
pixel_values torch.Size([3, 896, 896]) torch.float32

Nombre de tokens avec loss active (réponse) : 152
Tokens de la réponse cible (décodés) :
{"image_quality": "moyenne", "limitations": "câbles, flou", "visual_evidence": "Opacités diffuses et bilatérales dans les poumons, présence de nombreux câbles et dispositifs médicaux.", "justification": "L'image montre des opacités diffuses et bilatérales dans les poumons, ce qui suggère une condition pulmonaire. La présence de nombreux câbles et dispositifs médicaux rend l'interprétation difficile et peut masquer des détails importants.", "predicted_class": "suspected_opacity", "confidence": 0.75, "warning": "Avertissement: Incertitude IA, nécessite une relecture médicale"}<end_of_turn>



### Data collator

In [ ]:
def collate_fn(batch):

    if len(batch) == 1:
        item = batch[0]
        return {k: v.unsqueeze(0) for k, v in item.items()}

    max_len = max(item["input_ids"].shape[0] for item in batch)
    pad_id = processor.tokenizer.pad_token_id or 0

    input_ids, attention_mask, labels, pixel_values = [], [], [], []
    for item in batch:
        pad_len = max_len - item["input_ids"].shape[0]
        input_ids.append(torch.nn.functional.pad(item["input_ids"], (0, pad_len), value=pad_id))
        attention_mask.append(torch.nn.functional.pad(item["attention_mask"], (0, pad_len), value=0))
        labels.append(torch.nn.functional.pad(item["labels"], (0, pad_len), value=-100))
        pixel_values.append(item["pixel_values"])

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_mask),
        "labels": torch.stack(labels),
        "pixel_values": torch.stack(pixel_values),
    }

### Entraînement

`per_device_train_batch_size=1` + `gradient_accumulation_steps=8` pour simuler un batch effectif de 8 sans dépasser la VRAM. `gradient_checkpointing=True` pour économiser encore de la mémoire (au prix d'un entraînement un peu plus lent).

In [ ]:
OUTPUT_DIR = "lora_medgemma_uncertain"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
)

dataset_train = DatasetRadioFT(df_train, "dataset_Dev/images/", processor)

trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=dataset_train,
    data_collator=collate_fn,
)

trainer.train()

Step,Training Loss
5,0.370399
10,0.410175
15,0.324239
20,0.239348
25,0.210633
30,0.227613
35,0.162415
40,0.180107
45,0.156648


TrainOutput(global_step=45, training_loss=0.25350858370463053, metrics={'train_runtime': 802.9137, 'train_samples_per_second': 0.441, 'train_steps_per_second': 0.056, 'total_flos': 5334044896376928.0, 'train_loss': 0.25350858370463053, 'epoch': 3.0})

### Sauvegarde des adapters LoRA

In [ ]:
ADAPTER_DIR = "lora_medgemma_uncertain/adapter_final"

model_lora.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

print(f"Adapters LoRA sauvegardés dans : {ADAPTER_DIR}")

Adapters LoRA sauvegardés dans : lora_medgemma_uncertain/adapter_final


### Test

In [ ]:
model_lora.eval()
model_lora.config.use_cache = True

resultats_val = []

for index, row in df_val.iterrows():
    patient_ID = row["patientId"]
    chemin_Image = "dataset_Dev/images/" + patient_ID + ".dcm"
    print(f"[{index + 1}/{len(df_val)}] {patient_ID} (vérité: {row['class_origine']})")

    try:
        image = formatage_Image_dcm(chemin_Image).convert("RGB")
        messages = creer_Prompt_CoT()
        messages[0]["content"][0]["image"] = image

        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        ).to(model_lora.device, dtype=torch.bfloat16)

        input_len = inputs["input_ids"].shape[-1]
        with torch.inference_mode():
            outputs = model_lora.generate(**inputs, max_new_tokens=1200, do_sample=False)
            outputs = outputs[0][input_len:]

        reponse_brute = processor.decode(outputs, skip_special_tokens=True)
        reponse_json = extraire_json(reponse_brute)

        resultats_val.append({
            "patientId": patient_ID,
            "class_origine": row["class_origine"],
            "reponse_ia_brute": reponse_json,
        })
    except Exception as e:
        print(f"Erreur sur {patient_ID} : {e}")
        resultats_val.append({
            "patientId": patient_ID,
            "class_origine": row["class_origine"],
            "reponse_ia_brute": None,
        })

df_resultats_val = pd.DataFrame(resultats_val)

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[78/30] b6c45f07-6068-46af-b35d-302cd05ed4d2 (vérité: suspected_opacity)


c:\Users\doria\OneDrive\Documents\Python\SolutionDelivery\env\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[45/30] 195ddf15-9620-4827-97da-01a44d5842f7 (vérité: suspected_opacity)


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


...

[51/30] fabcaf4a-8724-4d33-848a-541457221d89 (vérité: uncertain)


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[106/30] 81c65b76-5600-4c2d-97ee-4281405f20b5 (vérité: uncertain)


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[83/30] 084aa98a-91aa-45d8-aa52-4fad8344b0bf (vérité: suspected_opacity)


### Classification report

In [ ]:
classes_predites_val = []
for _, row in df_resultats_val.iterrows():
    try:
        classes_predites_val.append(json.loads(row["reponse_ia_brute"]).get("predicted_class"))
    except (json.JSONDecodeError, TypeError):
        classes_predites_val.append("ERREUR_JSON")

df_resultats_val["predicted_class"] = classes_predites_val
df_json_valide_val = df_resultats_val[df_resultats_val["predicted_class"] != "ERREUR_JSON"]

labels = ["normal", "suspected_opacity", "uncertain"]
print(f"JSON valide : {len(df_json_valide_val)}/{len(df_resultats_val)} "
      f"({100*len(df_json_valide_val)/len(df_resultats_val):.1f}%)\n")
print(classification_report(df_json_valide_val["class_origine"], df_json_valide_val["predicted_class"],
                             labels=labels, digits=3))
print("Matrice de confusion (lignes=vrai, colonnes=prédit,", labels, "):")
print(confusion_matrix(df_json_valide_val["class_origine"], df_json_valide_val["predicted_class"], labels=labels))

JSON valide : 30/30 (100.0%)

                   precision    recall  f1-score   support

           normal      0.889     0.800     0.842        10
suspected_opacity      0.667     0.800     0.727        10
        uncertain      0.556     0.500     0.526        10

         accuracy                          0.700        30
        macro avg      0.704     0.700     0.699        30
     weighted avg      0.704     0.700     0.699        30

Matrice de confusion (lignes=vrai, colonnes=prédit, ['normal', 'suspected_opacity', 'uncertain'] ):
[[8 0 2]
 [0 8 2]
 [1 4 5]]
